# 🔌CONEXÃO
Estabelecer conexão com os bancos

In [ ]:
import psycopg2
from psycopg2.extras import DictCursor
import fdb
from dotenv import load_dotenv
import os

load_dotenv()

# Conexões
cnx_dest = fdb.connect(
    user=os.getenv("FDB_USER"),
    password=os.getenv("FDB_PASS"),
    host=os.getenv("FDB_HOST"),
    port=int(os.getenv("FDB_PORT")),
    database=os.getenv("FDB_PATH"),
    charset="WIN1252"
)
cur_dest = cnx_dest.cursor()


cnx_orig = psycopg2.connect(
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASS"),
    host=os.getenv("PG_HOST"),
    database=os.getenv("PG_DB"),
    options="-c search_path={}".format(os.getenv("PG_SCHEMA"))
)
cnx_orig.autocommit = True
cur_orig = cnx_orig.cursor(cursor_factory=DictCursor)

def commit():
    cnx_dest.commit()

# 🛠️ FERRAMENTAS
Funções, variáveis cache, hashmaps

In [ ]:
global cadest, empresa, exercicio
cadest = {}
empresa = cur_dest.execute("SELECT empresa FROM cadcli").fetchone()[0]
exercicio = cur_dest.execute("SELECT mexer FROM cadcli").fetchone()[0]

def limpa_tabela(tabelas):
    for tabela in tabelas:
        cur_dest.execute(f"DELETE FROM {tabela}")
    commit()

def cria_coluna(tabela, coluna):
    try:
        cur_dest.execute(f"ALTER TABLE {tabela} ADD {coluna} VARCHAR(255)")
    except fdb.DatabaseError as e:
        print(f"Erro ao criar coluna {coluna} na tabela {tabela}: {e}")
    else:
        commit()

dict_modalidades = {
"1":{"Licit": "MAT / SERV - CONCORRENCIA", 
      "Modlic": "CON4",
      "Codmod": 4},
"2":{"Licit": "MAT / SERV - CONCORRENCIA", 
      "Modlic": "CON4",
      "Codmod": 4},
"3":{"Licit": "MAT / SERV - TOMADA", 
      "Modlic": "TOM3",
      "Codmod": 3},
"4":{"Licit": "MAT / SERV - CONVITE", 
      "Modlic": "CS01",
      "Codmod": 7},
"5":{"Licit": "PREGÃO ELETRÔNICO", 
      "Modlic": "PE01",
      "Codmod": 9},
"6":{"Licit": "PREGÃO PRESENCIAL", 
      "Modlic": "PP01",
      "Codmod": 8},
"8":{"Licit": "LEILÃO", 
      "Modlic": "LEIL",
      "Codmod": 6},
"10":{"Licit": "DISPENSA", 
      "Modlic": "DI01",
      "Codmod": 1},
"11":{"Licit": "INEXIGIBILIDADE", 
      "Modlic": "IN01",
      "Codmod": 5},
"15":{"Licit": "DISPENSA ELETRÔNICA", 
      "Modlic": "DE01",
      "Codmod": 11},
"19":{"Licit": "DIÁLOGO COMPETITIVO", 
      "Modlic": "DC01",
      "Codmod": 28},
"20":{"Licit": "CONCURSO", 
      "Modlic": "CS01",
      "Codmod": 7},
"101":{"Licit": "LEILÃO ELETRÔNICO", 
      "Modlic": "LEIE",
      "Codmod": 20},
"102":{"Licit": "DIÁLOGO COMPETITIVO ELETRÔNICO", 
      "Modlic": "DCE",
      "Codmod": 29},
}

def to_cp1252_safe(value, replace_with=''):
    """
    Converte uma string para cp1252 de forma segura.
    Remove ou substitui caracteres inválidos.

    :param value: valor a converter
    :param replace_with: string usada no lugar de caracteres inválidos (default = '')
    :return: valor limpo
    """
    if isinstance(value, str):
        try:
            # Tenta converter diretamente
            value.encode('cp1252')
            return value
        except UnicodeEncodeError:
            # Remove/substitui caracteres inválidos
            return value.encode('cp1252', errors='replace').decode('cp1252').replace('?', replace_with)
    return value

try:
      cadest = {k: v for k,v in cur_dest.execute("select codreduz, max(cadpro) from cadest group by codreduz").fetchall()}
      cadest_ant = {k: v for k,v in cur_dest.execute("select codant, max(cadpro) from cadest group by codant").fetchall()}
except fdb.DatabaseError as e:
      pass

try:
    unidades = {k: v for k, v in cur_dest.execute("SELECT codant, sigla FROM cadunimedida").fetchall()}
except:
    pass


# 🛒 COMPRAS
<p>Extração, tratamento e carregamento dos dados referentes ao módulo compras</p>

In [ ]:
cur_dest.execute("""
execute block as
		begin
		DELETE FROM ICADREQ;
		DELETE FROM REQUI;
		DELETE FROM ICADPED;
		DELETE FROM CADPED;
		DELETE FROM regpreco;
		DELETE FROM regprecohis;
		DELETE FROM regprecodoc;
		DELETE FROM CADPRO_SALDO_ANT;
		DELETE FROM CADPROLIC_DETALHE_FIC;
		DELETE FROM CADPRO;
		DELETE FROM CADPRO_FINAL;
		DELETE FROM CADPRO_LANCE;
		DELETE FROM CADPRO_PROPOSTA;
		DELETE FROM PROLICS;
		DELETE FROM PROLIC;
		DELETE FROM CADPRO_STATUS;
		DELETE FROM CADLIC_SESSAO;
		DELETE FROM CADPROLIC_DETALHE;
		DELETE FROM CADPROLIC;
		DELETE FROM CADLIC;
		DELETE FROM VCADORC;
		DELETE FROM FCADORC;
		DELETE FROM ICADORC;
		DELETE FROM CADORC;
		DELETE FROM CADEST;
		DELETE FROM CENTROCUSTO;
		DELETE FROM DESTINO;
		DELETE FROM DESFORCRC_PADRAO;
		end;
""")
commit()

## CADASTROS BASE

### CADUNIMEDIDA

In [ ]:
limpa_tabela(("cadunimedida",))
cria_coluna("cadunimedida", "codant")
cria_coluna("cadunimedida", "sigla_ant")

insert = cur_dest.prep("""
    INSERT INTO cadunimedida(sigla, descricao, sigla_ant, codant)
    VALUES (?, ?, ?, ?)
""")

cur_orig.execute("""
with unidades_repetidas as (select
    substr(des_unidade, 1, 30) descricao,
    trim(substr(unid_sigla, 1, 5)) sigla,
    row_number() over (partition by substr(unid_sigla, 1, 5) order by id) rn,
    id codant,
    unid_sigla sigla_ant
from
    sicomv3_vargem.ac_unidade_medida)
select sigla, descricao, sigla_ant, codant from unidades_repetidas where rn = 1
order by codant
""")

repetidos = {}

for row in cur_orig.fetchall():
    try:
        sigla = to_cp1252_safe(row["sigla"])[:5]
        descricao = to_cp1252_safe(row["descricao"])
        sigla_ant = to_cp1252_safe(row["sigla_ant"])

        cur_dest.execute(insert, (sigla, descricao, sigla_ant, row["codant"]))
    except Exception as e:
        print(f"Error occurred while inserting row: {row}. Error: {e}")
commit()

unidades = {k: v for k, v in cur_dest.execute("SELECT codant, sigla FROM cadunimedida").fetchall()}

### GRUPO E SUBGRUPO

In [ ]:
limpa_tabela(("cadsubgr", "cadgrupo"))
cria_coluna("cadsubgr", "codant")
cria_coluna("cadsubgr", "codtce")

cur_orig.execute("""
select
	lpad(id::text, 3, '0') grupo,
	'000' subgrupo,
	des_grupo nome,
	id codant,
	case
		sta_ativo when 'S' then 'N'
		else 'S'
	end ocultar
from
	sicomv3_vargem.ac_grupos
union all
select
	lpad(grup_id::text, 3, '0') grupo,
	lpad(replace(cod_sub_grupo::text, '.', ''), 3, '0') subgrupo,
	t.des_subgrupo nome,
	t.id codant,
    case
		sta_ativo when 'S' then 'N'
		else 'S'
	end ocultar
from
	sicomv3_vargem.ac_subgrupos t
order by 1,2
""")

for row in cur_orig:
	try:
		if row['subgrupo'] == '000':
			cur_dest.execute("insert into cadgrupo(grupo, nome, ocultar) values(?, ?, ?)", (row['grupo'], row['nome'][:45], row['ocultar']))
		else:
			cur_dest.execute("insert into cadsubgr(grupo, subgrupo, nome, codant, ocultar) values(?, ?, ?, ?,? )", (row['grupo'], row['subgrupo'], row['nome'][:45], row['codant'], row['ocultar']))
		commit()
	except Exception as e:
		print(f"Erro ao inserir grupo/subgrupo {row['grupo']}/{row['subgrupo']}: {e}")


cur_dest.execute("insert into cadgrupo(grupo, nome) values ('000', 'SEM IDENTIFICAÇÃO')")
commit()

cur_dest.execute("insert into cadsubgr(grupo, subgrupo, nome) values ('000', '000', 'SEM IDENTIFICAÇÃO')")
commit()

### CADEST

In [ ]:
limpa_tabela(("cadest",))
cria_coluna("cadest", "key")
cria_coluna("cadest", "codant")

insert = cur_dest.prep("insert into cadest(grupo, subgrupo, codigo, cadpro, codreduz, disc1, ocultar, unid1, tipopro, usopro, key, codant) values (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)")

cur_orig.execute("""
select
	lpad(mat.grup_id::text, 3, '0') grupo,
	lpad(replace(cod_sub_grupo::text, '.', ''), 3, '0') subgrupo,
	trim(cod_material) codreduz,
	des_resumida disc1,
	case
        mat.sta_ativo when 'S' then 'N'
        else 'S'
    end ocultar,
    substr(unid_sigla, 1, 5) sigla,
     case 
        sta_servico when 'N' then 1
        else 3 end tipo,
	mat.id::int codant
from
	sicomv3_vargem.ac_materiais mat
left join sicomv3_vargem.ac_unidade_medida med on
	med.id = mat.unme_id
	and med.inst_id = mat.inst_id
join sicomv3_vargem.ac_subgrupos sub on
	sub.id = mat.sugr_id
order by
	mat.grup_id ,
	sugr_id,
	seq_material
""")

subgrupos_desdobrados = {k:[v, 0] for k, v in cur_dest.execute("select grupo||'.'||subgrupo, cast(subgrupo as integer) subgrupo_int from cadsubgr")}
max_subgrupo_grupo = {k: v for k, v in cur_dest.execute("select grupo, max(cast(subgrupo as integer)) from cadsubgr group by 1")}

tipo_uso_produto = {
	1: ['P', 'C'],
	2: ['P', 'P'],
	3: ['S', '']
}

for row in cur_orig.fetchall():
	key = f"{row['grupo']}.{row['subgrupo']}"
	subgrupo_int, codigo = subgrupos_desdobrados[key]
	codigo += 1

	if codigo > 999:
		subgrupo_ant = max_subgrupo_grupo[row['grupo']]
		subgrupo_int = max_subgrupo_grupo[row['grupo']] + 1
		codigo = 1
		subgrupos_desdobrados[key] = [subgrupo_int, codigo]
		max_subgrupo_grupo[row['grupo']] = subgrupo_int
	else:
		subgrupos_desdobrados[key][1] = codigo

	tipopro, usopro = tipo_uso_produto.get(row['tipo'], ['P', 'C'])

	cadpro = f'{row["grupo"]}.{subgrupo_int:03}.{codigo:03}'

	cur_dest.execute(insert, (row['grupo'], f'{subgrupo_int:03}', f'{codigo:03}', cadpro, row['codreduz'], row['disc1'][:1024], 'N', row['sigla'], tipopro, usopro, key, row['codant']))
commit()

cur_dest.execute("""
INSERT INTO CADSUBGR (grupo, subgrupo, nome)
WITH CTE AS (
SELECT DISTINCT a.grupo, a.subgrupo, KEY FROM cadest a
WHERE NOT EXISTS (SELECT 1 FROM cadsubgr x WHERE a.grupo = x.grupo AND a.SUBGRUPO = x.SUBGRUPO))
SELECT a.GRUPO, a.SUBGRUPO, b.NOME FROM cte a
JOIN CADSUBGR b ON b.grupo||'.'||b.SUBGRUPO = a."KEY" 
""")
commit()


In [ ]:
cadest = {k: v for k,v in cur_dest.execute("select codreduz, max(cadpro) from cadest group by codreduz").fetchall()}
cadest_ant = {k: v for k,v in cur_dest.execute("select codant, max(cadpro) from cadest group by codant").fetchall()}

### ALMOXARIFADOS

In [ ]:
limpa_tabela(("destino",))
cria_coluna("destino", "codant")

insert = cur_dest.prep("""insert into destino(cod,desti,empresa,codant) values(?,?,?,?)""")

cur_orig.execute("""
select
	u.des_unidade desti ,
	lpad(d.id::text, 9, '0') cod,
	d.inst_id empresa
from
	sicomv3_vargem.ac_unidade_almoxarifado d
left join sicomv3_vargem.ac_unidade u on
	u.id = d.unid_id
""")

for row in cur_orig.fetchall():
    cur_dest.execute(insert, (row['cod'], row['desti'], row['empresa'], row['codant']))

cur_dest.execute(insert, ('000000000', 'CONVERSAO', empresa, None))
commit()

### CENTRO DE CUSTO

In [ ]:
limpa_tabela(("centrocusto",))
cria_coluna("centrocusto", "codant")

try:
    cur_dest.execute(f"insert into destino(cod,desti,empresa) values('000000000','CONVERSAO',{empresa})")
except fdb.DatabaseError as e:
    print(f"Erro ao inserir destino: {e}")
else:
    commit()
    
local = cur_dest.execute("select first 1 poder, orgao, unidade from desorc where empresa = (select empresa from cadcli)").fetchonemap()
if not local:
    local = {
        'poder': '02',
        'orgao': '01',
    }

insert = cur_dest.prep("""
    insert into centrocusto (codccusto, descr, ccusto, ocultar, empresa, poder, orgao, destino, codant) values (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
select
    u.id::int codccusto,
    substr(u.des_unidade,1,60) descr,
    '001' ccusto,
    case
        sta_ativo when 'N' then 'S'
        else 'N'
    end ocultar,
    coalesce(lpad(d.id::text, 9, '0'), '000000000') destino,
    u.cod_unidade codant
from
    sicomv3_vargem.ac_unidade u
left join sicomv3_vargem.ac_unidade_almoxarifado d on
    u.id = d.unid_id
""")

for row in cur_orig.fetchall():
    try:
        cur_dest.execute(insert, (row['codccusto'], to_cp1252_safe(row['descr']), '001', row['ocultar'], empresa, local['poder'], local['orgao'], row['destino'], to_cp1252_safe(row['codant'])))
    except Exception as e:
        print(f"Erro ao inserir centro de custo {row['codccusto']}: {e}")
cur_dest.execute(insert, (0, 'CONVERSÃO', '001', 'N', empresa, local['poder'], local['orgao'], '000000000', None))
commit()

In [ ]:
# Armazena hashmap de centros de custo
centros_de_custo = {k: v for k, v in cur_dest.execute("select codant, codccusto from centrocusto").fetchall()}



## COTAÇÃO



In [ ]:
cur_dest.execute("""
execute block as
    begin
    DELETE FROM vcadorc;
    DELETE FROM fcadorc;
    DELETE FROM icadorc;
    DELETE FROM cadorc;
end;
""")
commit()

### CADORC

In [ ]:
limpa_tabela(("cadorc",))
cria_coluna("cadorc", "codant")

insert = cur_dest.prep("""
insert into cadorc
	(id_cadorc,
	num,
	ano,
	numorc,
	dtorc,
	descr,
	prioridade,
	obs,
	status,
	liberado,
	codccusto,
	liberado_tela,
	empresa,
	codant) values
	(?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
--COTACAO
select
	to_char(cotac.num_cotacao, 'fm00000/')|| to_char(cotac.ano_cotacao%2000, 'fm00') numorc,
	cotac.dat_criacao dtorc,
	substr(cotac.des_objeto, 1, 1024) descr,
	cotac.des_observacao obs,
	case
		when sta_cotacao = 'A' then 'AB'
		when sta_cotacao = 'N' then 'CA' --cancelada
		when sta_cotacao = 'P' then 'CO' --em processo
		else 'EC'
	end status,
	u.cod_unidade codccusto,
	'C'||cotac.id::int codant
from
	siop_vargem.pc_cotacao cotac
left join sicomv3_vargem.ac_unidade u on
	cotac.unid_id = u.id
union all
--SOLICITACAO
select
	9|| to_char(sol.num_solicitacao, 'fm0000/')|| to_char(sol.ano_solicitacao%2000, 'fm00') numorc,
	sol.dat_criacao dtorc,
	substr(sol.des_objeto, 1, 1024) descr,
	sol.des_justificativa obs,
	case
		when sta_solicitacao = 'A' then 'AB'
		when sta_solicitacao = 'N' then 'CA' --cancelada
		when sta_solicitacao = 'P' then 'CO' --em processo
		else 'EC'
	end status,
	u.cod_unidade codccusto,
	'S'||sol.id::int codant
from
	siop_vargem.pc_solicitacao_compra sol
left join sicomv3_vargem.ac_unidade u on
	sol.unid_id = u.id
where not exists (select 1 from siop_vargem.pc_cotacao_solicitacao x where x.soco_id = sol.id)
""")

for i, row in enumerate(cur_orig):
	num, ano = row['numorc'].split('/')
	i += 1

	if row['status'] == 'LC':
		liberado = 'S'
		liberado_tela = 'L'

	elif row['status'] == 'EC':
		liberado = 'S'
		liberado_tela = 'L'

	else:
		liberado = 'N'
		liberado_tela = 'N'

	try:
		codccusto = centros_de_custo.get(row['codccusto'], 0)

		cur_dest.execute(insert, (
			i,
			num,
			ano,
			row['numorc'],
			row['dtorc'],
			row['descr'],
			'NORMAL',
			row['obs'],
			row['status'],
			liberado,
			codccusto,
			liberado_tela,
			empresa,
			row['codant']
		))
	except Exception as e:
		print(f"Error: {e} - id: {row['codant']}")
commit()

In [ ]:
cotacoes_solicitacoes = {k: v for k, v in cur_dest.execute("select codant, cast(id_cadorc as integer) from cadorc").fetchall()}

### ICADORC

In [ ]:
limpa_tabela(("icadorc",))
cria_coluna("icadorc", "itemcota")
cria_coluna("icadorc", "itemid")

insert = cur_dest.prep("""
insert into icadorc (
    numorc,
    item,
    cadpro,
    qtd, 
    valor, 
    itemorc, 
    codccusto, 
    itemorc_ag,
    id_cadorc)
values(?,?,?,?,?,?,?,?,?)
""")

cur_orig.execute("""
with COTACAO as
(select
	to_char(cotac.num_cotacao, 'fm00000/')|| to_char(cotac.ano_cotacao%2000, 'fm00') numorc,
	row_number() over (partition by cotac.id order by 1) item,
	trim(cod_material) codreduz,
	sum(soli.qtd_item) qtd,
	0 valor,
	u.cod_unidade codccusto,
	'C'||cotac.id::int codant
from
	siop_vargem.pc_cotacao cotac
join siop_vargem.pc_cotacao_item cotaci
	on cotac.id = cotaci.cota_id 
join siop_vargem.pc_solicitacao_item soli
	on soli.id = cotaci.soit_id
left join sicomv3_vargem.ac_unidade u on
	cotac.unid_id = u.id
where cotaci.sta_cotar = 'S' --and cotac.id = 3
group by 1,3,5,6,7, soli.des_resumida, cotac.id
order by cotac.id, soli.des_resumida),
SOLICITACAO as 
(select
	9|| to_char(sol.num_solicitacao, 'fm0000/')|| to_char(sol.ano_solicitacao%2000, 'fm00') numorc,
	item.seq_item::int item ,
	trim(cod_material) codreduz,
	item.qtd_item qtd,
	0 valor,
	u.cod_unidade codccusto,
	'S'||sol.id::int codant
from
	siop_vargem.pc_solicitacao_compra sol
join siop_vargem.pc_solicitacao_item item on
	item.soco_id = sol.id
left join sicomv3_vargem.ac_unidade u on
	sol.unid_id = u.id
where not exists (select 1 from siop_vargem.pc_cotacao_solicitacao x where x.soco_id = sol.id)
order by sol.id)
select * from COTACAO
union all 
select * from SOLICITACAO
""")

for row in cur_orig:
    try:
        cadpro = cadest[row['codreduz']]
        codccusto = centros_de_custo.get(row['codccusto'], 0)
        idcadorc = cotacoes_solicitacoes[row['codant']]

    except Exception as e:
        print(f"Erro codant {row['codant']}: {e}")
        break

    else:
        cur_dest.execute(insert, (
            row['numorc'],
            row['item'],
            cadpro,
            row['qtd'],
            row['valor'],
            row['item'],
            codccusto,
            row['item'],
            idcadorc,
        ))
commit()    

### FCADORC

In [ ]:
limpa_tabela(("fcadorc",))

insert = cur_dest.prep("insert into fcadorc(numorc, codif, nome, valorc, id_cadorc) values (?,?,?,?,?)")

cur_orig.execute("""
select
	pess.pess_nom_contato nome,
	pess.pess_id::int codif,
	valorc,
	'C'||pess.cota_id::int idcadorc,
	to_char(cotac.num_cotacao, 'fm00000/')|| to_char(cotac.ano_cotacao%2000, 'fm00') numorc
from
	siop_vargem.pc_cotacao_pessoa pess
join siop_vargem.pc_cotacao cotac
	on cotac.id = pess.cota_id
join (select
		cotacp.cota_id,
		pcp.pess_id::int codif,
		sum(vlr_total) valorc
	from
		siop_vargem.pc_cotacao_proposta cotacp
	join siop_vargem.pc_cotacao_pessoa pcp
		on pcp.id = cotacp.cope_id 
	group by 1,2) valores
	on valores.cota_id = pess.cota_id and valores.codif = pess.pess_id
""")

for row in cur_orig:
	idcadorc = cotacoes_solicitacoes[row['idcadorc']]

	cur_dest.execute(insert, (
		row['numorc'],
		row['codif'],
		row['nome'][:70],
		row['valorc'],
		idcadorc
	))
commit()

### VCADORC

In [ ]:
limpa_tabela(("vcadorc",))

insert = cur_dest.prep("""insert into vcadorc(numorc, item, codif, vlruni, vlrtot, ganhou, vlrganhou, classe, id_cadorc) values (?,?,?,?,?,?,?,?,?)""")

cur_orig.execute("""
with itens as (
SELECT
    cotaci.cota_id::int,
   	dense_rank() over (partition by cotaci.cota_id order by soli.des_resumida) item,
    cotaci.id::int
FROM siop_vargem.pc_cotacao cotac
JOIN siop_vargem.pc_cotacao_item cotaci
    ON cotac.id = cotaci.cota_id
JOIN siop_vargem.pc_solicitacao_item soli
    ON soli.id = cotaci.soit_id
WHERE cotaci.sta_cotar = 'S'),
ganhadores as (
	select
		cope_id,
		coit_id ,
		propg.cota_id ,
		pess1.pess_id codif
	from
		siop_vargem.pc_cotacao_proposta propg
	join siop_vargem.pc_cotacao_pessoa pess1 on
		pess1.id = propg.cope_id
	where
		sta_julgamento = 'V'
),
propostas AS (
    SELECT
        to_char(cotac.num_cotacao, 'fm00000/') ||
        to_char(cotac.ano_cotacao % 2000, 'fm00') numorc,
        i.item,
        pess.pess_id::int codif,
        TRUNC(AVG(prop.vlr_unitario)::numeric, 2) vlruni,
        SUM(prop.vlr_total) vlrtot,
        CASE WHEN dipr_id = 1 THEN 'UN' ELSE 'GL' END classe,
        'C' || cotac.id::int codant,
        cotac.id id_cotacao
    FROM siop_vargem.pc_cotacao_proposta prop
    JOIN siop_vargem.pc_cotacao_pessoa pess
      ON prop.cope_id = pess.id
    JOIN siop_vargem.pc_cotacao cotac
      ON cotac.id = prop.cota_id
    JOIN itens i
      ON i.cota_id = prop.cota_id
     AND prop.coit_id::int = i.id
    LEFT JOIN ganhadores gan
      ON gan.coit_id = prop.coit_id
     AND gan.cota_id = prop.cota_id
    WHERE pess.pess_id = gan.codif
    GROUP BY
        1,2,3,6,7,cotac.id
)
SELECT
    p.*,
    FIRST_VALUE(codif) OVER (
        PARTITION BY id_cotacao, item
        ORDER BY vlruni
    ) AS ganhou,
    FIRST_VALUE(vlruni) OVER (
        PARTITION BY id_cotacao, item
        ORDER BY vlruni
    ) AS vlrganhou
FROM propostas p;
""")

for row in cur_orig:
    idcadorc = cotacoes_solicitacoes[row['codant']]

    cur_dest.execute(insert, (
        row['numorc'],
        row['item'],
        row['codif'],
        row['vlruni'],
        row['vlrtot'],
        row['ganhou'],
        row['vlrganhou'],
        row['classe'],
        idcadorc
    ))
commit()

## LICITAÇÕES

In [ ]:
#LIMPA LICITAÇÕES
cur_dest.execute("""
execute block as
    begin
    DELETE FROM regpreco;
    DELETE FROM regprecohis;
    DELETE FROM regprecodoc;
    DELETE FROM CADPROLIC_DETALHE_FIC;
    DELETE FROM CADPRO;
    DELETE FROM CADPRO_FINAL;
    DELETE FROM CADPRO_LANCE;
    DELETE FROM CADPRO_PROPOSTA;
    DELETE FROM PROLICS;
    DELETE FROM PROLIC;
    DELETE FROM CADPRO_STATUS;
    DELETE FROM CADLIC_SESSAO;
    DELETE FROM CADPROLIC_DETALHE;
    DELETE FROM CADPROLIC;
    DELETE FROM CADLIC;
    end;
""")
commit()

### CADLIC

In [ ]:
limpa_tabela(("cadlic",))

insert = cur_dest.prep("""
INSERT
        INTO
        cadlic(
        numlic,
        proclic,
        numero,
        ano,
        comp,
        licnova,
        liberacompra,
        discr,
        registropreco,
        microempresa,
        numpro,
        discr7,
        datae,
        processo_data,
        dtadj,
        dthom,
        codtce,
        anomod,
        modlic,
        licit,
        codmod,
        dtpub,
        dtenc,
        empresa,
        processo,
        processo_ano,
        dtreal,
        numorc,
        id_cadorc,
        tpapostilamento)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
    ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
    ?, ?, 'I')
""")

cur_orig.execute("""
select
    row_number() over(partition by pro.ano_processo, pro.num_processo 
            order by ano_processo, num_processo, dat_cancelamento desc) sequencia,
    pro.id::int numlic,
    to_char(pro.num_processo::int, 'fm000000/')|| pro.ano_processo::int%2000 proclic,
    pro.ano_processo::int ano,
    case
        when pro.sta_processo = 'A' then 1
        when pro.sta_processo = 'J' then 2
        when pro.sta_processo = 'N' then 4
        else 3
    end comp,
    1 licnova,
    substr(pro.des_objeto, 1, 1024) discr,
    pro.des_justificativa objeto,
    pro.num_modalidade detalhe,
    (
    select
        sol.sta_abertura_rp rp
    from
        siop_vargem.pc_processo_solicitacao pc
    join siop_vargem.pc_solicitacao_compra sol on
        sol.id = pc.soco_id
    where
        pc.prco_id = pro.id
    limit 1) registropreco,
    3 microempresa,
    pro.num_modalidade::int numpro,
    'Menor Preco Unitario' discr7,
    pro.dat_proc_licitatorio datae,
    pro.dat_adjudicacao dtadj,
    pro.dat_homologacao dthom,
    nullif(pro.cod_audesp, 0) codtce,
    pro.ano_modalidade::int anomod,
    pro.moli_id::int::text modlic,
    modali.des_modalidade descricaomodalidade,
    pro.dat_julg_audesp datajultamento,
    pro.dat_ata_aber_habi datahabilitacao,
    pro.dat_encerramento dtenc,
    pro.num_processo_adm processo,
    nullif(pro.ano_processo_adm, 0)::int processo_ano
from
    siop_vargem.pc_processo_compra pro
left join siop_vargem.pc_modalidade_licitacao modali on
    modali.id = pro.moli_id
order by numlic
""")

for row in cur_orig:
    if row['sequencia'] > 1: 
            continue
    
    try:
        info_modalidades = dict_modalidades.get(row['modlic'], {"Licit": "DISPENSA", "Modlic": "DI01", "Codmod": 1})

        licit = info_modalidades["Licit"]
        modlic = info_modalidades["Modlic"]
        codmod = info_modalidades["Codmod"]

        cur_dest.execute(insert, (
            row['numlic'], 
            row['proclic'],
            row['numlic'],
            row['ano'],
            row['comp'],
            row['licnova'],
            'S',
            row['discr'],
            row['registropreco'],
            row['microempresa'],
            row['numpro'],
            row['discr7'],
            row['datae'],
            row['datae'],
            row['dtadj'],
            row['dthom'],
            row['codtce'],
            row['ano'],
            modlic,
            licit,
            codmod,
            row['datae'],
            row['dthom'],
            empresa,
            row['processo'],
            row['processo_ano'],
            row['datajultamento'],
            None,
            None
        ))
    except Exception as e:
        print(f"Erro ao inserir licitação {row['numlic']}: {e}")
commit()

cur_dest.execute("""UPDATE CADLIC SET FK_MODLICANO = (SELECT PK_MODLICANO FROM MODLICANO WHERE CODMOD = CADLIC.CODMOD AND ANOMOD = CADLIC.ANO AND CADLIC.EMPRESA = MODLICANO.EMPRESA) WHERE CODMOD IS NOT NULL""")
commit()

cur_dest.execute('ALTER TRIGGER TBU_CADLICITACAO INACTIVE')
commit()

cur_dest.execute("""
EXECUTE BLOCK AS
BEGIN
	UPDATE cadlicitacao SET nlicitacao = nlicitacao * 10000;
	UPDATE cadlicitacao SET nlicitacao = numlic;
END;
""")
commit()

cur_dest.execute('ALTER TRIGGER TBU_CADLICITACAO ACTIVE')
commit()

### CADLIC SESSAO

In [ ]:
cur_dest.execute("""INSERT INTO CADLIC_SESSAO (NUMLIC, SESSAO, DTREAL, HORREAL, COMP, DTENC, HORENC, SESSAOPARA, MOTIVO) 
SELECT L.NUMLIC, CAST(1 AS INTEGER), L.DTREAL, L.HORREAL, L.COMP, L.DTENC, L.HORENC, CAST('T' AS VARCHAR(1)), CAST('O' AS VARCHAR(1)) FROM CADLIC L 
WHERE numlic not in (SELECT FIRST 1 S.NUMLIC FROM CADLIC_SESSAO S WHERE S.NUMLIC = L.NUMLIC)""")
commit()

### PARTICIPANTES

In [ ]:
limpa_tabela(("prolics","prolic"))

insert_prolic = cur_dest.prep("insert into prolic(numlic,codif,nome,status) values(?,?,?,?)")
insert_prolics = cur_dest.prep("insert into prolics(sessao,numlic,codif,habilitado,status,cpf,representante) values(?,?,?,?,?,?,?)")

cur_orig.execute("""
select
	row_number() over(partition by pro.id, pess.pess_id order by pro.id, pess.pess_id ) sequencia,
	pro.id::int numlic,
	pess.pess_id::int codif,
	pess.pess_nom_contato nome,
	pess.num_cpf cpf,
	substr(pess.nom_repres, 1, 100) representante
from
	siop_vargem.pc_processo_pessoa pess
join siop_vargem.pc_processo_compra pro on
	pro.id = pess.prco_id
order by
	pro.id,
	pess.pess_id
""")

for row in cur_orig:
    cur_dest.execute(insert_prolic, (row['numlic'], row['codif'], to_cp1252_safe(row['nome'][:40]), 'A'))
    cur_dest.execute(insert_prolics, (1, row['numlic'], row['codif'], 'S', 'A', row['cpf'], to_cp1252_safe(row['representante'])))
commit()

### CADPROLIC

In [ ]:
limpa_tabela(("cadprolic_detalhe_fic", "cadprolic_detalhe", "cadprolic"))
cria_coluna("cadprolic", "codant")

insert_cadprolic = cur_dest.prep("""
INSERT INTO cadprolic(item,
        item_mask,
        numorc,
        itemorc,
        cadpro,
        quan1,
        vamed1,
        vatomed1,
        codccusto,
        reduz,
        numlic,
        microempresa,
        item_lc147,
        tlance)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,'$')
""")

cur_orig.execute("""
with ITENS_COTACAO as
(select
	to_char(cotac.num_cotacao, 'fm00000/')|| to_char(cotac.ano_cotacao%2000, 'fm00') numorc,
	row_number() over (partition by cotac.id order by 1) item,
	trim(cod_material) codreduz,
	sum(soli.qtd_item) qtd,
	0 valor,
	u.cod_unidade codccusto,
	cotac.id::int codant
from
	siop_vargem.pc_cotacao cotac
join siop_vargem.pc_cotacao_item cotaci
	on cotac.id = cotaci.cota_id 
join siop_vargem.pc_solicitacao_item soli
	on soli.id = cotaci.soit_id
left join sicomv3_vargem.ac_unidade u on
	cotac.unid_id = u.id
where cotaci.sta_cotar = 'S' --and cotac.id = 3
group by 1,3,5,6,7, soli.des_resumida, cotac.id
order by cotac.id, soli.des_resumida),
ITENS_PROC as (
SELECT
    ppi.prco_id,
    trim(psi.cod_material) AS codreduz,
    MAX(ppi.soco_id) AS soco_id,
    max(case when ppi.qtd_cota_reservada <> 0 then seq_cota_reservada else null end) AS item_cota,
    SUM(ppi.qtd_cota_principal) AS qtd_cota_principal,
    SUM(ppi.qtd_cota_reservada) AS qtd_cota_reservada
FROM siop_vargem.pc_processo_item ppi
JOIN siop_vargem.pc_solicitacao_item psi
    ON psi.id = ppi.soit_id
GROUP BY
    ppi.prco_id,
    trim(psi.cod_material),
    ppi.seq_cota_principal
ORDER BY
    ppi.seq_cota_principal)
select row_number() over (partition by prco_id order by ic.item) item, numorc, item itemorc, ic.codreduz, coalesce(nullif(ip.qtd_cota_principal,0), qtd) quan1, nullif(ip.qtd_cota_reservada,0) qtd_reserva, ip.prco_id::int numlic, codccusto, ip.item_cota::int from ITENS_PROC ip
join siop_vargem.pc_cotacao_solicitacao pcs 
	on pcs.soco_id = ip.soco_id
join ITENS_COTACAO ic
	on ic.codant = pcs.cota_id and ic.codreduz = ip.codreduz
""")

for row in cur_orig:
    try:
        cadpro = cadest[row['codreduz']]
        codccusto = centros_de_custo.get(row['codccusto'], 0)

        if row['item_cota']:
            item_lc147 = row['item_cota']

            cur_dest.execute(insert_cadprolic, (row['item_cota'], row['item_cota'], row['numorc'], row['itemorc'], cadpro, row['qtd_reserva'], 0, 0, codccusto, 'N', row['numlic'], 'S', row['item']))

        cur_dest.execute(insert_cadprolic, (row['item'], row['item'], row['numorc'], row['itemorc'], cadpro, row['quan1'], 0, 0, codccusto, 'N', row['numlic'], 'N', None))
    
    except Exception as e:
        print(f"Erro ao inserir cadprolic {row['numlic']} - {row['item']}: {e}")
commit()

cur_dest.execute("""
    MERGE INTO cadprolic a USING (SELECT numorc, item, vlrganhou, id_cadorc FROM vcadorc WHERE codif = ganhou) b
    ON a.numorc = b.numorc AND a.itemorc = b.item
    WHEN MATCHED THEN UPDATE SET a.vamed1 = b.vlrganhou, a.vatomed1 = a.quan1*b.vlrganhou, a.id_cadorc = b.id_cadorc
""")
commit()

cur_dest.execute("""
    INSERT INTO CADPROLIC_DETALHE (NUMLIC,item,CADPRO,quan1,VAMED1,VATOMED1,marca,CODCCUSTO,ITEM_CADPROLIC, item_lc147,id_cadorc,itemorc,numorc)
    select numlic, item, cadpro, quan1, vamed1, vatomed1, marca, codccusto, item, item_lc147, b.ID_CADORC, itemorc, numorc from cadprolic b where
    not exists (select 1 from cadprolic_detalhe c where b.numlic = c.numlic and b.item = c.item);
""")
commit()

cur_dest.execute("""
    insert into 
    cadprolic_detalhe_fic (numlic, item, codigo, qtd, valor, qtdadt, valoradt, codccusto, qtdmed, valormed, tipo) 
    select numlic, item, item, quan1, vatomed1, quan1, vatomed1, codccusto, quan1, vatomed1, 'C' from cadprolic b where
    not exists (select 1 from cadprolic_detalhe_fic c where b.numlic = c.numlic and b.item = c.item);
""")
commit()

cur_dest.execute("""INSERT INTO cadlotelic (descr, lotelic, numlic) SELECT distinct 'Lote ' || lotelic, lotelic, numlic FROM 
    cadprolic a WHERE lotelic IS NOT NULL AND NOT EXISTS ( SELECT 1 FROM CADLOTELIC c WHERE c.numlic = a.numlic AND a.lotelic = c.lotelic)""")
commit()

### PROPOSTA

In [ ]:
limpa_tabela(("cadpro", "cadpro_final", "cadpro_proposta"))

insert = cur_dest.prep("""
INSERT
    INTO
    cadpro_proposta(sessao,
    codif,
    item,
    itemp,
    quan1,
    vaun1,
    vato1,
    numlic,
    status,
    subem,
    marca,
    itemlance)
VALUES(?,?,?,?,?,?,?,?,?,?,?,?)""")

cur_orig.execute("""
with CADPROLIC as (
	with ITENS_COTACAO as
    (select
        to_char(cotac.num_cotacao, 'fm00000/')|| to_char(cotac.ano_cotacao%2000, 'fm00') numorc,
        row_number() over (partition by cotac.id order by 1) item,
        trim(cod_material) codreduz,
        sum(soli.qtd_item) qtd,
        0 valor,
        u.cod_unidade codccusto,
        cotac.id::int codant
    from
        siop_vargem.pc_cotacao cotac
    join siop_vargem.pc_cotacao_item cotaci
        on cotac.id = cotaci.cota_id 
    join siop_vargem.pc_solicitacao_item soli
        on soli.id = cotaci.soit_id
    left join sicomv3_vargem.ac_unidade u on
        cotac.unid_id = u.id
    where cotaci.sta_cotar = 'S' --and cotac.id = 3
    group by 1,3,5,6,7, soli.des_resumida, cotac.id
    order by cotac.id, soli.des_resumida),
    ITENS_PROC as (
    SELECT
        ppi.prco_id,
        trim(psi.cod_material) AS codreduz,
        MAX(ppi.soco_id) AS soco_id,
        max(case when ppi.qtd_cota_reservada <> 0 then seq_cota_reservada else null end) AS item_cota,
        SUM(ppi.qtd_cota_principal) AS qtd_cota_principal,
        SUM(ppi.qtd_cota_reservada) AS qtd_cota_reservada
    FROM siop_vargem.pc_processo_item ppi
    JOIN siop_vargem.pc_solicitacao_item psi
        ON psi.id = ppi.soit_id
    GROUP BY
        ppi.prco_id,
        trim(psi.cod_material),
        ppi.seq_cota_principal
    ORDER BY
        ppi.seq_cota_principal)
    select row_number() over (partition by prco_id order by ic.item) item, numorc, item itemorc, ic.codreduz, coalesce(nullif(ip.qtd_cota_principal,0), qtd) quan1, nullif(ip.qtd_cota_reservada,0) qtd_reserva, ip.prco_id::int numlic, codccusto, ip.item_cota::int from ITENS_PROC ip
    join siop_vargem.pc_cotacao_solicitacao pcs 
        on pcs.soco_id = ip.soco_id
    join ITENS_COTACAO ic
        on ic.codant = pcs.cota_id and ic.codreduz = ip.codreduz
)
select
	ppp.prco_id::int numlic,
	ppp.seq_item::int item,
	pess.pess_id::int codif,
	sum(ppp.qtd_item) quan1,
	ppp.vlr_unitario vaun1,
	sum(ppp.vlr_total) vato1,
	case
		ppp.sta_julgamento when 'V' then 1
		else 2
	end subem,
	substr(ppp.des_marca,1,50) marca
from
	siop_vargem.pc_processo_proposta ppp
join siop_vargem.pc_processo_pessoa pess on
	pess.id = ppp.prpe_id
where
	coalesce(ppp.sta_compativel_mercado, 'N') = 'S'
	and not exists (select 1 from siop_vargem.pc_processo_lote_item x where x.prpr_id = ppp.id)
group by 1,2,3,5,7,8
union all
--Itens de Lotes
select
	ppp.prco_id::int numlic,
	c.item,
	pess.pess_id::int codif,
	sum(ppli.qtd_item) quan1,
	ppli.vlr_unitario vaun1,
	sum(ppli.vlr_total) vato1,
	case
		ppp.sta_julgamento when 'V' then 1
		else 2
	end subem,
	substr(ppli.des_marca,1,50) marca
from
	siop_vargem.pc_processo_lote_item ppli
join siop_vargem.pc_processo_proposta ppp
	on ppp.id = ppli.prpr_id
join siop_vargem.pc_processo_pessoa pess on
	pess.id = ppp.prpe_id
join siop_vargem.pc_processo_item ppi
	on ppi.id = ppli.prit_id
join (select distinct id, trim(cod_material) codreduz from siop_vargem.pc_solicitacao_item) psi
	on psi.id = ppi.soit_id
join CADPROLIC c 
	on c.codreduz = psi.codreduz 
	and c.numlic =  ppp.prco_id::int
group by 1,2,3,5,7,8
order by 1,2,3
""")           

for row in cur_orig:
    try:
        cur_dest.execute(insert, (1, row['codif'], row['item'], row['item'], row['quan1'], row['vaun1'], row['vato1'], row['numlic'], 'C', row['subem'], row['marca'], 'S'))
    except Exception as e:
        print(f"Error inserting row: {row['numlic']}, {row['codif']}, {row['item']} - {e}")
commit()

cur_dest.execute("""
    MERGE INTO cadlic a USING (SELECT numlic, max(id_cadorc) id_cadorc, max(numorc) numorc FROM cadprolic WHERE id_cadorc IS NOT NULL GROUP BY 1) b
    ON a.numlic = b.numlic
    WHEN MATCHED THEN UPDATE SET a.id_cadorc = b.id_cadorc, a.numorc = b.numorc;
""")
commit()

cur_dest.execute("""
    MERGE INTO cadorc a USING (SELECT max(numlic) numlic, max(proclic) proclic, id_cadorc FROM cadlic WHERE id_cadorc IS NOT NULL GROUP BY id_cadorc) b
    ON a.id_cadorc = b.id_cadorc
    WHEN MATCHED THEN UPDATE SET a.numlic = b.numlic, a.proclic = b.proclic;
""")
commit()

cur_dest.execute("""
    MERGE INTO cadlic a USING (SELECT id_cadorc, sum(v.vlrtot) total FROM VCADORC v WHERE ganhou = codif GROUP BY 1) b
    ON a.id_cadorc = b.id_cadorc
    WHEN MATCHED THEN UPDATE SET a.valor = b.total, a.valor1 = b.total
""")
commit()

cur_dest.execute("""update cadlic set comp = 3 where exists (select 1 from cadpro_proposta x where x.numlic = cadlic.numlic)""")
commit()

cur_dest.execute("""
    INSERT INTO cadpro_proposta(sessao, codif, item, itemp, quan1, vaun1, vato1, numlic, status, subem, marca, itemlance)
    SELECT 1 sessao, a.codif, ITEM, item itemp, c.QTD, a.VLRUNI, a.VLRTOT, b.NUMLIC, 'C' status, 1 subem, NULL marca, 'S' itemlance FROM VCADORC a
    JOIN cadorc b USING (id_cadorc)
    JOIN icadorc c USING (id_cadorc, item)
    JOIN cadlic d USING (numlic) 
    WHERE NOT EXISTS (SELECT 1 FROM cadpro_proposta x WHERE x.NUMLIC = b.NUMLIC) AND a.codif = a.GANHOU AND codmod in (1,5) AND comp = 3
""")
commit()

### CADPRO LANCE

In [ ]:
cur_dest.execute("""insert into cadpro_lance (sessao, rodada, codif, itemp, vaunl, vatol, status, subem, numlic)
	SELECT sessao, 1 rodada, CODIF, ITEMP, VAUN1, VATO1, 'F' status, SUBEM, numlic FROM CADPRO_PROPOSTA cp where subem = 1 and not exists
	(select 1 from cadpro_lance cl where cp.codif = cl.codif and cl.itemp = cp.itemp and cl.numlic = cp.numlic)""")
commit()

### CADPRO FINAL

In [ ]:
cur_dest.execute("""INSERT into cadpro_final (numlic, ult_sessao, codif, itemp, vaunf, vatof, STATUS, subem)
	SELECT numlic, sessao, codif, itemp, vaun1, vato1, CASE WHEN status = 'F' THEN 'C' ELSE status end, subem FROM cadpro_proposta
	WHERE NOT EXISTS (SELECT 1 FROM cadpro_final f WHERE f.numlic = cadpro_proposta.numlic AND f.itemp = cadpro_proposta.itemp AND f.codif = cadpro_proposta.codif)""")
commit()

cur_dest.execute("""update cadlic a SET comp = 3 WHERE comp = 1 AND EXISTS (SELECT 1 FROM cadpro_proposta x WHERE subem = 1 AND x.numlic = a.numlic) AND NOT EXISTS (SELECT 1 FROM cadpro x WHERE x.numlic = a.numlic)""")
commit()

### CADPRO_STATUS

In [ ]:
cur_dest.execute("""
    INSERT INTO cadpro_status (numlic, sessao, itemp, item, telafinal)
    SELECT b.NUMLIC, 1 AS sessao, a.item, a.item, 'I_ENCERRAMENTO'
    FROM CADPROLIC a
    JOIN cadlic b ON a.NUMLIC = b.NUMLIC
    WHERE NOT EXISTS (
        SELECT 1
        FROM cadpro_status c
        WHERE a.numlic = c.numlic
    )
    AND EXISTS (
        SELECT 1
        FROM cadlic d
        WHERE d.numlic = a.numlic AND d.comp = 3
    );
""")
commit()

### CADPRO

In [ ]:
cria_coluna("cadpro", "codant")

cur_dest.execute("""INSERT INTO CADPRO (
  CODIF, CADPRO, QUAN1, VAUN1, VATO1, SUBEM, STATUS,
  ITEM, NUMORC, ITEMORCPED, CODCCUSTO, FICHA, ELEMENTO, DESDOBRO,
  NUMLIC, ULT_SESSAO, ITEMP, QTDADT, QTDPED, VAUNADT, VATOADT,
  PERC, QTDSOL, ID_CADORC, VATOPED, VATOSOL, TPCONTROLE_SALDO,
  QTDPED_FORNECEDOR_ANT, VATOPED_FORNECEDOR_ANT, MARCA
)
WITH dados AS (
  SELECT
    a.CODIF,
      c.CADPRO,
      c.ITEM,
      d.NUMORC,
      c.CODCCUSTO,
      c.FICHA,
      c.ELEMENTO,
      c.DESDOBRO,
      a.NUMLIC,
      b.ITEMP,
      d.ID_CADORC,
      a.marca,
      a.quan1  AS qtdunit,
      a.VAUN1 AS vaun,
      a.vato1 AS vatotal
  FROM CADPRO_PROPOSTA a
  JOIN CADPRO_STATUS b
    ON b.NUMLIC = a.NUMLIC
    AND b.ITEMP  = a.ITEMP
    AND b.SESSAO = a.SESSAO
  JOIN CADPROLIC_DETALHE c
    ON c.NUMLIC = a.NUMLIC
    AND b.ITEMP  = c.ITEM_CADPROLIC
  JOIN CADLIC d
    ON d.NUMLIC = a.NUMLIC
  WHERE a.SUBEM = 1
    AND a.STATUS = 'C'
)
SELECT
    CODIF,
    CADPRO,
    qtdunit,
    vaun,
    vatotal,
    1,
    'C',
    ITEM,
    NUMORC,
    ITEM,
    CODCCUSTO,
    FICHA,
    ELEMENTO,
    DESDOBRO,
    NUMLIC,
    1,
    ITEMP,
    qtdunit,
    0,
    vaun,
    vatotal,
    0,
    0,
    ID_CADORC,
    0,
    0,
    'Q',
    0,
    0,
    marca
FROM dados d
WHERE NOT EXISTS (
    SELECT 1
    FROM CADPRO cp
    WHERE cp.NUMLIC = d.NUMLIC
      AND cp.ITEM   = d.ITEM
      AND cp.CODIF  = d.CODIF
)""")
commit()

### REGPRECO

In [ ]:
cur_dest.execute("""
EXECUTE BLOCK AS  
    BEGIN  
    INSERT INTO REGPRECODOC (NUMLIC, CODATUALIZACAO, DTPRAZO, ULTIMA)  
    SELECT DISTINCT A.NUMLIC, 0, DATEADD(1 YEAR TO A.DTHOM), 'S'  
    FROM CADLIC A WHERE A.REGISTROPRECO = 'S' AND A.DTHOM IS NOT NULL  
    AND NOT EXISTS(SELECT 1 FROM REGPRECODOC X  
    WHERE X.NUMLIC = A.NUMLIC);  

    INSERT INTO REGPRECO (COD, DTPRAZO, NUMLIC, CODIF, CADPRO, CODCCUSTO, ITEM, CODATUALIZACAO, QUAN1, VAUN1, VATO1, QTDENT, SUBEM, STATUS, ULTIMA)  
    SELECT B.ITEM, DATEADD(1 YEAR TO A.DTHOM), B.NUMLIC, B.CODIF, B.CADPRO, B.CODCCUSTO, B.ITEM, 0, B.QUAN1, B.VAUN1, B.VATO1, 0, B.SUBEM, B.STATUS, 'S'  
    FROM CADLIC A INNER JOIN CADPRO B ON (A.NUMLIC = B.NUMLIC) WHERE A.REGISTROPRECO = 'S' AND A.DTHOM IS NOT NULL  
    AND NOT EXISTS(SELECT 1 FROM REGPRECO X  
    WHERE X.NUMLIC = B.NUMLIC AND X.CODIF = B.CODIF AND X.CADPRO = B.CADPRO AND X.CODCCUSTO = B.CODCCUSTO AND X.ITEM = B.ITEM);  

    INSERT INTO REGPRECOHIS (NUMLIC, CODIF, CADPRO, CODCCUSTO, ITEM, CODATUALIZACAO, QUAN1, VAUN1, VATO1, SUBEM, STATUS, MOTIVO, MARCA, NUMORC, ULTIMA)  
    SELECT B.NUMLIC, B.CODIF, B.CADPRO, B.CODCCUSTO, B.ITEM, 0, B.QUAN1, B.VAUN1, B.VATO1, B.SUBEM, B.STATUS, B.MOTIVO, B.MARCA, B.NUMORC, 'S'  
    FROM CADLIC A INNER JOIN CADPRO B ON (A.NUMLIC = B.NUMLIC) WHERE A.REGISTROPRECO = 'S' AND A.DTHOM IS NOT NULL  
    AND NOT EXISTS(SELECT 1 FROM REGPRECOHIS X  
    WHERE X.NUMLIC = B.NUMLIC AND X.CODIF = B.CODIF AND X.CADPRO = B.CADPRO AND X.CODCCUSTO = B.CODCCUSTO AND X.ITEM = B.ITEM);  
END;""")
commit()

## PEDIDOS

### ANTERIORES AO EXERCÍCIO

In [ ]:
limpa_tabela(("cadpro_saldo_ant", "regpreco_saldo_ant"))

insert = cur_dest.prep("""
insert into cadpro_saldo_ant (
    ano,
    numlic,
    item,
    cadpro,
    qtdped,
    vatoped)
values (?,?,?,?,?,?)
""")

cur_orig.execute(f"""
select
	emp.prco_id::int numlic,
	item.num_seq::int item,
	cast(item.mate_id as integer) material,
	u.cod_unidade::int codccusto,
	sum(item.qtd_item) qtdped,
	max(item.vlr_unitario) vatoped
from
	siop_vargem.pc_solicitacao_empenho emp
join siop_vargem.pc_solicitacao_empenho_item item on
	item.soem_id = emp.id
join siop_vargem.pc_processo_compra pro on
	pro.id = emp.prco_id
join siop_vargem.pc_processo_item proitem on
	proitem.id = item.prit_id
	and proitem.prco_id = pro.id
left join siop_vargem.pc_solicitacao_compra sol on
	sol.id = proitem.soco_id
left join sicomv3_vargem.ac_unidade u on
	u.id = sol.unid_id
where
	emp.ano_solicitacao < {exercicio}
	and emp.dat_cancelamento is null
	and item.num_seq is not null
group by 1, 2, 3, 4
""")

for row in cur_orig:
    cadpro = cadest_ant[str(row['material'])]
    try:
        cur_dest.execute(insert, (
			int(exercicio)-1,
			row['numlic'],
			row['item'],
			cadpro,
			row['qtdped'],
			row['vatoped']
		))
        commit()
    except fdb.DatabaseError as e:
        if e.args[1] == -803:
            cur_dest.execute("update cadpro_saldo_ant set qtdped = qtdped + ?, vatoped = vatoped + ? where numlic = ? and item = ?", (row['qtdped'], row['vatoped'], row['numlic'], row['item']))
            commit()
        else:
            raise

### DO EXERCÍCIO

In [ ]:
limpa_tabela(("icadped where id_cadped in (SELECT id_cadped FROM cadped where id_cadpedlicit is not null)", "cadped where id_cadpedlicit is not null","icadped", "cadped"))

itens_processo = {k:[v1,v2] for k,v1,v2 in cur_dest.execute("select numlic||'.'||codif||'.'||codant, cadpro, item from cadpro").fetchall()}
itens_processo_sem_codif = {k:[v1,v2] for k,v1,v2 in cur_dest.execute("select numlic||'.'||codant, cadpro, item from cadpro a").fetchall()}

inser_cadped = cur_dest.prep("insert into cadped (numped, num, ano, datped, codif, id_cadped, empresa, numlic, obs, codccusto, dt_anulacao, id_cadpedlicit, npedlicit) values (?,?,?,?,?,?,?,?,?,?,?,?,?)")
insert_icadped = cur_dest.prep("insert into icadped (numped, item, cadpro, qtd, prcunt, prctot, codccusto, id_cadped, item_licit) values (?,?,?,?,?,?,?,?,?)")

cur_orig.execute(f"""
select
	to_char(emp.cod_solicitacao, 'fm00000/') || emp.ano_solicitacao::int%2000 numped,
	emp.dat_emissao datped,
	emp.pess_id::int codif,
	emp.id::int id_cadped,
	emp.prco_id::int numlic,
	coalesce(u.cod_unidade, '0') codccusto,
	emp.dat_cancelamento dt_anulacao,
	emp.sta_soli_empenho status,
	row_number() over(partition by emp.id order by item.num_seq) item,
	item.num_seq::int item_licit,
	item.vlr_unitario prcunt,
	cast(item.mate_id as integer) material,
	emp.id empid,
	sum(item.qtd_item) qtd,
	(
	select
		dot.num_ficha
	from
		siop_vargem.pc_solicitacao_empenho_dotacao empdot
	join siop_vargem.of_dotacao dot on
		dot.id_dotacao = empdot.dota_id
	where
		empdot.soem_id = emp.id
		and empdot.soei_id = max(item.id)
		and empdot.ano_base = {exercicio}
		and empdot.suel_id is not null
	limit 1) as ficha
from
	siop_vargem.pc_solicitacao_empenho emp
join siop_vargem.pc_solicitacao_empenho_item item on
	item.soem_id = emp.id
left join siop_vargem.pc_processo_compra pro on
	pro.id = emp.prco_id
left join siop_vargem.pc_processo_item proitem on
	proitem.id = item.prit_id
	and proitem.prco_id = pro.id
left join siop_vargem.pc_solicitacao_compra sol on
	sol.id = proitem.soco_id
left join sicomv3_vargem.ac_unidade u on
	u.id = sol.unid_id
where
	emp.ano_solicitacao = {exercicio}
group by 1,2,3,4,5,6,7,8,10,11,12,13, item.num_seq
order by
	id_cadped
""")

id_cadped = 0

for row in cur_orig:
    if row['id_cadped'] != id_cadped:
        id_cadped = row['id_cadped']
        num, ano = row['numped'].split('/')
        codccusto = centros_de_custo.get(row['codccusto'], 0)
        
        cur_dest.execute(inser_cadped, (
            row['numped'],
            num,
            exercicio,
            row['datped'],
            row['codif'],
            id_cadped,
            empresa,
            row['numlic'],
            None,
            codccusto,
            row['dt_anulacao'],
            None,
            None
        ))
        commit()
    
    try:
        cur_dest.execute(insert_icadped, (
            row['numped'],
            row['item'],
            cadest_ant[str(row['material'])],
            row['qtd'],
            row['prcunt'],
            row['prcunt'] * row['qtd'],
            codccusto,
            id_cadped,
            row['item_licit']
        ))
    except Exception as e:
        print(f'Erro ao inserir pedido {row["numped"]}: {e}')
        continue
commit()

cur_dest.execute("""
UPDATE cadped a SET a.codatualizacao_rp = (SELECT max(codatualizacao) 
FROM regprecodoc x WHERE x.numlic = a.numlic)
""")
commit()

cur_dest.execute("""
UPDATE CADPED A SET PROCLIC = (SELECT proclic FROM CADLIC X WHERE X.NUMLIC = A.NUMLIC);
""")
commit()

### AUTORIZAÇÃO

In [ ]:
limpa_tabela(("icadped where id_cadped in (SELECT id_cadped FROM cadped where id_cadpedlicit is not null)", "cadped where id_cadpedlicit is not null"))

cur_orig.execute(f"""
with itens_pai as (
    select distinct
    scit_id,
    item.mate_id,
    row_number() over(partition by item.soem_id order by num_seq) item,
    cast(mate_id as integer) material,
    num_seq item_licit,
    coalesce(u.cod_unidade, '0') codccusto,
    emp.prco_id::int numlic,
    emp.pess_id::int codif
from
    siop_vargem.pc_solicitacao_empenho emp
join siop_vargem.pc_solicitacao_empenho_item item on
    item.soem_id = emp.id
left join siop_vargem.pc_processo_compra pro on
    pro.id = emp.prco_id
left join siop_vargem.pc_processo_item proitem on
    proitem.id = item.prit_id
    and proitem.prco_id = pro.id
left join siop_vargem.pc_solicitacao_compra sol on
    sol.id = proitem.soco_id
left join sicomv3_vargem.ac_unidade u on
    u.id = sol.unid_id
where emp.sta_soli_empenho <> 'C'
)
select
	psc.id,
    9||to_char(pa.num_autorizacao, 'fm0000/') || pa.ano_autorizacao::int%2000 numped,
    pa.dat_emissao datped,
    coalesce(pse.pess_id::int, codif) codif,
    numlic,
    des_objeto obs,
    codccusto,
    pa.dat_cancelamento dt_anulacao,
    pse.id::int id_cadpedlicit,
    to_char(pse.cod_solicitacao, 'fm00000/') || pse.ano_solicitacao::int%2000 npedlicit,
    item::int item,
    material,
    psci.qtd_item qtd,
    vlr_unitario prcunt,
    psci.qtd_item * psci.vlr_unitario prctot,
    ip.item_licit::int item_licit
from
    siop_vargem.pc_solicitacao_consumo psc
left join siop_vargem.pc_soli_consumo_item psci
    on psci.socn_id = psc.id 
left join siop_vargem.pc_autorizacao pa 
    on pa.socn_id = psc.id
join siop_vargem.pc_solicitacao_empenho pse
    on pse.socn_id = psc.id
    and sta_soli_empenho <> 'C'
join itens_pai ip
    on ip.scit_id = psci.id
where pa.ano_autorizacao = {exercicio} --and 9||to_char(pa.num_autorizacao, 'fm0000/') || pa.ano_autorizacao::int%2000 = '90033/26'
order by pse.id::int, item::int
""")

id_cadped = cur_dest.execute("select max(id_cadped)+1 from cadped").fetchone()[0]
numped_ant = ''

for row in cur_orig:
    codif = row['codif']
    codccusto = centros_de_custo.get(row['codccusto'], 0)
    
    if row['numped'] != numped_ant:
        id_cadped += 1
        num, ano = row['numped'].split('/')
        
        cur_dest.execute(inser_cadped, (
            row['numped'],
            num,
            exercicio,
            row['datped'],
            codif,
            id_cadped,
            empresa,
            row['numlic'],
            row['obs'],
            codccusto,
            row['dt_anulacao'],
            row['id_cadpedlicit'],
            row['npedlicit']
        ))
        commit()

        numped_ant = row['numped']
    
    try:
        cur_dest.execute(insert_icadped, (
            row['numped'],
            row['item'],
            cadest_ant[str(row['material'])],
            row['qtd'],
            row['prcunt'],
            row['prctot'],
            codccusto,
            id_cadped,
            row['item_licit']
        ))
    except Exception as e:
        print(f'Erro ao inserir pedido {row["numped"]}: {e}')
        continue
commit()



# 📄 CONTRATOS

## CADASTROS

In [ ]:
limpa_tabela(('contratos',))

insert = cur_dest.prep("""
INSERT
	INTO
	contratos (codigo,
	contratonum,
	empresa,
	ano,
	dtassi,
	vigeni,
	vigenf,
	dtpubl,
	veicpub,
	nproli,
	flegal,
	fundlegal,
	codif,
	objeti,
	objeto,
	objeto_completo,
	garant,
	valcon,
	tipo_contrato,
	ataregpreco,
	tipoco,
	proclic,
	numajuste) values (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
select distinct	
	to_char(pc.id, 'fm0000/')||pc.ano_contrato::int%2000 codigo,
	to_char(pc.num_contrato, 'fm0000/')||pc.ano_contrato::int%2000 contratonum,
	pc.ano_contrato::int ano,
	pc.dat_assinatura dtassi,
	pc.dat_vigencia_ini vigeni,
	pc.dat_vigencia_fim vigenf,
	pc.dat_assinatura dtpubl,
	pro.num_processo::int nproli,
	case when nullif(pc.prco_id,0) is not null then 'LICITAÇÃO' else null end fundlegal,
	coalesce(ppe.pess_id, pc.pess_id_saldo)::int codif,
	pc.des_objeto objeti,
	pc.des_objeto objeto,
	pc.des_objeto objeto_completo,
	'SG' garant,
	valcon,
	case when pc.tip_contrato = 'D' then 'P' else 'R' end tipo_contrato,
	'N' ataregpreco,
	99 tipoco,
	to_char(pro.num_processo::int, 'fm000000/')|| pro.ano_processo::int%2000 proclic,
	nullif(pc.cod_audesp,0) numajuste	
from
	siop_vargem.pc_contrato pc
left join siop_vargem.pc_processo_compra pro 
	on pro.id = pc.prco_id
left join siop_vargem.pc_processo_pessoa ppe
	on ppe.id = pc.prpe_id
left join (select cont_id, sum(qtd_item*pci.vlr_unitario) valcon from siop_vargem.pc_contrato_item pci group by 1) itens
	on itens.cont_id = pc.id
""")

for row in cur_orig:
    cur_dest.execute(insert, (row['codigo'], row['contratonum'], empresa, row['ano'], row['dtassi'], row['vigeni'], row['vigenf'], row['dtpubl'], None, row['nproli'], row['fundlegal'], row['fundlegal'], row['codif'], to_cp1252_safe(row['objeti'][:60]), to_cp1252_safe(row['objeto'][:60]), to_cp1252_safe(row['objeto_completo']), row['garant'], row['valcon'], row['tipo_contrato'], 'N', '99', row['proclic'], row['numajuste']))
commit()

cur_dest.execute("""
MERGE INTO contratos a USING (SELECT proclic, substring(licit from 1 FOR 20) licit, registropreco, numpro FROM cadlic) b
ON a.proclic = b.proclic
WHEN MATCHED THEN UPDATE SET a.MODALI = b.licit, ataregpreco = registropreco, numlicmod = numpro
""") 
commit()

# cur_dest.execute("""
# merge into contratos a using (select codigo, sum(vadem-anula) empenhado from despes a
# join (select codigo, cast(substring(codigo from 1 for 4) as integer) id from contratos)
# on a.id_contrato_ant = id
# group by 1) b on a.codigo = b.codigo
# when matched then update set a.empenhado = b.empenhado
# """) 
# commit()

## ITENS

In [ ]:
limpa_tabela(["CONTRATOSITEMLICIT"])

insert = cur_dest.prep("""
INSERT INTO CONTRATOSITEMLICIT (CONTRATO, ITEM, LOTELIC, CADPRO, QTD, UND, DESCR, VLUNIT, VLTOTAL) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

cur_orig.execute("""
select
	to_char(pci.cont_id, 'fm0000/')||pc.ano_contrato::int%2000 contrato,
	row_number() over (partition by pci.cont_id order by pci.seq_item::int) item,
	pci.mate_id::int codreduz,
	pci.qtd_item qtd,
	pci.vlr_unitario vlunit,
	pci.qtd_item * pci.vlr_unitario vltotal
from
	siop_vargem.pc_contrato_item pci
join (select id, ano_contrato from siop_vargem.pc_contrato) pc
	on pc.id = pci.cont_id
""")

for row in cur_orig:
    cadpro = cadest_ant[str(row['codreduz'])]
    if not cadpro:
        print(f'Produto {row["codreduz"]} do contrato {row["codigo"]} não encontrado no cadastro de estoque.')
        continue

    try:
        cur_dest.execute(insert, (
            row['contrato'],
            row['item'],
            '',
            cadpro,
            row['qtd'],
            '', 
            '', 
            row['vlunit'],
            row['vltotal']
        ))
    except Exception as e:
        print(f'Erro ao inserir item contrato {row["contrato"]}: {e} - {row}')
        break
commit()

## ADITAMENTOS

In [ ]:
limpa_tabela(["contratosaditamento"])

cur_orig.execute("""
select
	to_char(pc.id, 'fm0000/')||pc.ano_contrato::int%2000 contrato,
	pc.data_efetivacao dtlan,
	pc.dat_encerramento dataencerramento,
	valor,
	pa.des_objeto descricao,
	pc.data_efetivacao dtpublicacao,
	pc.data_efetivacao datainsc,
	case 
		when pa.tiad_id in (2,4) then 'Acréscimo'
		when pa.tiad_id = 9 then 'Prazo / Valor / Reequilíbrio'
		when pa.tiad_id = 10 then 'Apost. Reajuste'
		when pa.tiad_id = 12 then 'Prorrogação'
	end tipohist,
	'Bilateral' tipoalt,
	to_char(pa.num_aditivo, 'fm00000/')||pa.ano_aditivo::int%2000 termo,
	pa.num_aditivo::int codigo
from
	siop_vargem.pc_aditivo pa
join siop_vargem.pc_contrato pc 
	on pc.id = pa.cont_id 
join (select adit_id, sum(vlr_unitario*qtd_item) valor from siop_vargem.pc_aditivo_item group by 1) pai
	on pai.adit_id = pa.id
order by cont_id, pa.id
""")

insert = cur_dest.prep("""
INSERT INTO CONTRATOSADITAMENTO (CONTRATO, DTLAN, dataencerramento, valor, descricao, datainsc, TIPOHIST, TIPOALT, termo) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

for row in cur_orig:
    try:
        cur_dest.execute(insert, (
            row['contrato'],
            row['dtlan'],
            row['dataencerramento'],
            row['valor'],
            row['descricao'][:240] if row['descricao'] else None,
            row['datainsc'],
            row['tipohist'],
            row['tipoalt'],
            row['termo']
        ))
    except Exception as e:
        print(e)
        break
commit()

## TERMO DE CIÊNCIA

In [ ]:
limpa_tabela(["cpfcontratos"])

insert = cur_dest.prep("""insert into cpfcontratos (codigo, contrato, cpf, nome, tipoassinatura, emailpro, emailpes, assinatura) values (?, ?, ?, ?, ?, ?, ?, ?)""")

cur_orig.execute("""
select
	pctc.id::int codigo,
	to_char(pc.id, 'fm0000/')||pc.ano_contrato::int%2000 contrato,
	num_cpf cpf,
	pctc.des_nome nome,
	sta_parte tipoassinatura,
	pctc.des_email_prof emailpro,
	pctc.des_email_pess emailpes,
	pctc.sta_assinou_contr assinatura
from
	siop_vargem.pc_contrato_termo_ciencia pctc
join siop_vargem.pc_contrato pc 
	on pc.id = pctc.cont_id 
""")

for row in cur_orig:
    cur_dest.execute(insert, (row['codigo'], row['contrato'], row['cpf'], row['nome'], row['tipoassinatura'], row['emailpro'], row['emailpes'], row['assinatura']))
commit()